In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


# 07 — Gold Processing
Per-instrument theme assignment (equity prefers industry_group over sector; ETF/Fund prefer category_group over category), removed-records detection, delisted flag, the 4 Gold tables, Theme Coverage Score, and a Delta time-travel demo on Gold.

In [0]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

financial_instrument = spark.read.format("delta").load(silver_path("financial_instrument"))
equity_ext = spark.read.format("delta").load(silver_path("equity_extension"))
etf_ext = spark.read.format("delta").load(silver_path("etf_extension"))
fund_ext = spark.read.format("delta").load(silver_path("fund_extension"))
dim_country = spark.read.format("delta").load(silver_path("dim_country"))
taxonomy_mapping = spark.read.format("delta").load(silver_path("taxonomy_mapping")).where("is_active = true")

print(f"financial_instrument: {financial_instrument.count()} rows")

financial_instrument: 66105 rows


In [0]:
def assign_theme(ext_df, asset_type, primary_field, secondary_field):
    pm = taxonomy_mapping.where((F.col("asset_type") == asset_type) & (F.col("source_field") == primary_field)) \
        .select(F.col("source_value").alias("_pv"), F.col("normalized_theme").alias("_pt"),
                F.col("confidence_score").alias("_pc"), F.col("mapping_method").alias("_pm"),
                F.col("taxonomy_version").alias("_pver"))
    sm = taxonomy_mapping.where((F.col("asset_type") == asset_type) & (F.col("source_field") == secondary_field)) \
        .select(F.col("source_value").alias("_sv"), F.col("normalized_theme").alias("_st"),
                F.col("confidence_score").alias("_sc"), F.col("mapping_method").alias("_sm"),
                F.col("taxonomy_version").alias("_sver"))

    joined = (
        ext_df
        .join(pm, ext_df[primary_field] == pm["_pv"], "left")
        .join(sm, ext_df[secondary_field] == sm["_sv"], "left")
    )

    use_primary = F.col("_pt").isNotNull()
    use_secondary = ~use_primary & F.col("_st").isNotNull()

    return joined.select(
        "instrument_id",
        F.lit(asset_type).alias("asset_type"),
        F.when(use_primary, F.lit(primary_field)).when(use_secondary, F.lit(secondary_field)).otherwise(F.lit(primary_field)).alias("source_field_used"),
        F.when(use_primary, F.col(primary_field)).when(use_secondary, F.col(secondary_field)).otherwise(F.col(primary_field)).alias("source_classification"),
        F.when(use_primary, F.col("_pt")).when(use_secondary, F.col("_st")).alias("normalized_theme"),
        F.when(use_primary, F.col("_pc")).when(use_secondary, F.col("_sc")).otherwise(F.coalesce(F.col("_pc"), F.col("_sc"))).alias("confidence_score"),
        F.when(use_primary, F.col("_pm")).when(use_secondary, F.col("_sm")).otherwise(F.coalesce(F.col("_pm"), F.col("_sm"))).alias("mapping_method"),
        F.coalesce(F.col("_pver"), F.col("_sver"), F.lit(TAXONOMY_VERSION)).alias("taxonomy_version"),
    )

instrument_theme = (
    assign_theme(equity_ext, "equity", "industry_group", "sector")
    .unionByName(assign_theme(etf_ext, "etf", "category_group", "category"))
    .unionByName(assign_theme(fund_ext, "fund", "category_group", "category"))
)

instrument_theme.write.format("delta").mode("overwrite").save(silver_path("instrument_theme_assignment"))
print(f"instrument_theme_assignment: {instrument_theme.count()} rows, {instrument_theme.where('normalized_theme IS NOT NULL').count()} with a real theme")

instrument_theme_assignment: 66105 rows, 43828 with a real theme


In [0]:
latest_per_type = financial_instrument.groupBy("asset_type").agg(F.max("snapshot_date").alias("latest_snapshot"))

financial_instrument_status = (
    financial_instrument
    .join(latest_per_type, on="asset_type", how="left")
    .withColumn("record_status", F.when(F.col("snapshot_date") == F.col("latest_snapshot"), F.lit("ACTIVE")).otherwise(F.lit("REMOVED_FROM_SOURCE")))
    .drop("latest_snapshot")
)

removed_count = financial_instrument_status.where("record_status = 'REMOVED_FROM_SOURCE'").count()
print(f"REMOVED_FROM_SOURCE: {removed_count} (expected 0 today)")

delisted_frames = []
for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
    if "equities" not in classes:
        continue
    path = latest_snapshot_path("equities", exch)
    if path is None:
        continue
    df = (
        spark.read.option("header", True).option("inferSchema", True)
        .option("multiLine", True).option("escape", "\"")
        .csv(f"{path}/*.csv")
        .select("symbol", F.coalesce(F.col("exchange"), F.lit(exch)).alias("exchange"), "delisted")
    )
    delisted_frames.append(df)

delisted_df = delisted_frames[0]
for f in delisted_frames[1:]:
    delisted_df = delisted_df.unionByName(f)

delisted_df = (
    delisted_df
    .withColumn("instrument_id", F.sha2(F.concat_ws("|", F.lit("equity"), "symbol", "exchange"), 256))
    .select("instrument_id", F.col("delisted").alias("is_delisted"))
)
print(f"Delisted equities flagged: {delisted_df.where('is_delisted = true').count()}")

REMOVED_FROM_SOURCE: 0 (expected 0 today)
Delisted equities flagged: 3779


In [0]:
gold_financial_instrument_catalog = (
    financial_instrument_status
    .join(instrument_theme.select("instrument_id", "normalized_theme", "confidence_score"), on="instrument_id", how="left")
    .join(dim_country.select(F.col("country_name").alias("country"), "region"), on="country", how="left")
    .join(delisted_df, on="instrument_id", how="left")
    .select(
        "instrument_id", "symbol",
        F.col("instrument_name").alias("name"),
        "asset_type", "country", "region", "exchange", "currency",
        "normalized_theme",
        F.col("confidence_score").alias("mapping_confidence"),
        "record_status",
        F.coalesce(F.col("is_delisted"), F.lit(False)).alias("is_delisted"),
    )
)
gold_financial_instrument_catalog.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path("financial_instrument_catalog"))
print(f"gold_financial_instrument_catalog: {gold_financial_instrument_catalog.count()} rows")

gold_market_universe_summary = (
    gold_financial_instrument_catalog.groupBy("country", "exchange", "asset_type").agg(F.count("*").alias("instrument_count"))
)
gold_market_universe_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path("market_universe_summary"))
print(f"gold_market_universe_summary: {gold_market_universe_summary.count()} rows")

gold_theme_universe = (
    gold_financial_instrument_catalog.where(F.col("normalized_theme").isNotNull())
    .groupBy("normalized_theme", "asset_type", "country", "exchange").agg(F.count("*").alias("instrument_count"))
)
gold_theme_universe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path("theme_universe"))
print(f"gold_theme_universe: {gold_theme_universe.count()} rows")

gold_financial_instrument_catalog: 66105 rows
gold_market_universe_summary: 388 rows
gold_theme_universe: 1734 rows


## Gold 4 + Theme Coverage Score
Weighted composite: asset-class breadth (40%, max=3), country reach (30%), exchange reach (20%), instrument scale (10%, capped at 1000).

In [0]:
theme_pivot = (
    gold_financial_instrument_catalog.where(F.col("normalized_theme").isNotNull())
    .groupBy("normalized_theme").pivot("asset_type", ["equity", "etf", "fund"]).agg(F.count(F.lit(1))).na.fill(0)
    .withColumnRenamed("equity", "equity_count").withColumnRenamed("etf", "etf_count").withColumnRenamed("fund", "fund_count")
)

theme_breadth = (
    gold_financial_instrument_catalog.where(F.col("normalized_theme").isNotNull())
    .groupBy("normalized_theme")
    .agg(
        F.countDistinct("asset_type").alias("asset_class_breadth"),
        F.countDistinct("country").alias("country_count"),
        F.countDistinct("exchange").alias("exchange_count"),
        F.count("*").alias("total_instruments"),
    )
)
max_countries = gold_financial_instrument_catalog.where(F.col("country").isNotNull()).select("country").distinct().count()
print(f"max_countries (denominator): {max_countries}")

gold_cross_asset_theme_new = (
    theme_pivot.join(theme_breadth, on="normalized_theme")
    .withColumn(
        "theme_coverage_score",
        F.round(
            (F.col("asset_class_breadth") / F.lit(3.0)) * 40 +
            (F.col("country_count") / F.lit(float(max_countries))) * 30 +
            (F.col("exchange_count") / F.lit(10.0)) * 20 +
            F.least(F.col("total_instruments") / F.lit(1000.0), F.lit(1.0)) * 10,
            2
        )
    )
)

gold_ct_path = gold_path("cross_asset_theme")
if DeltaTable.isDeltaTable(spark, gold_ct_path):
    target = DeltaTable.forPath(spark, gold_ct_path)
    (target.alias("t").merge(gold_cross_asset_theme_new.alias("s"), "t.normalized_theme = s.normalized_theme")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    gold_cross_asset_theme_new.write.format("delta").mode("overwrite").save(gold_ct_path)

gold_cross_asset_theme = spark.read.format("delta").load(gold_ct_path)
print(f"gold_cross_asset_theme: {gold_cross_asset_theme.count()} rows")
display(gold_cross_asset_theme.orderBy(F.desc("theme_coverage_score")))

history = spark.sql(f"DESCRIBE HISTORY delta.`{gold_ct_path}`")
display(history.select("version", "timestamp", "operation"))

versions = [r["version"] for r in history.select("version").collect()]
earliest_version = min(versions)
recovered = spark.read.format("delta").option("versionAsOf", earliest_version).load(gold_ct_path)
current = spark.read.format("delta").load(gold_ct_path)

print(f"Recovered version {earliest_version} — row count: {recovered.count()}")
print(f"Current version — row count: {current.count()}")

max_countries (denominator): 95
gold_cross_asset_theme: 9 rows


normalized_theme,equity_count,etf_count,fund_count,asset_class_breadth,country_count,exchange_count,total_instruments,theme_coverage_score
Financial Services,5643,1456,2743,3,70,10,9842,92.11
Consumer,6306,149,13,3,62,10,6468,89.58
Technology,5920,536,33,3,59,10,6489,88.63
Energy,1604,156,12,3,58,10,1772,88.32
Materials,5093,127,13,3,57,10,5233,88.0
Industrials,6400,175,6,3,57,10,6581,88.0
Healthcare,4015,160,45,3,48,10,4220,85.16
Utilities,904,55,3,3,46,10,962,84.15
Real Estate,1755,364,142,3,38,10,2261,82.0


version,timestamp,operation
1,2026-08-21T22:27:32.000Z,MERGE
0,2026-08-21T22:22:40.000Z,WRITE


Recovered version 0 — row count: 9
Current version — row count: 9
